For the lazy ones, check out [Wuerth's RED-Expert DC-DC Converter Designer](https://redexpert.we-online.com/we-redexpert/en/#/dcdc-converter-embedded)

In [36]:
# Design parameters
Vin = 12
Vout = 5
Vd = 0 # Schottky diode forward voltage drop (non-synchronous buck converter)
r = 0.3 # ripple current factor, 0.3 to 0.6 is typical
fsw = 750e3 # Hz
Iout = 2 # A
η = 0.9 # Efficiency
ΔIo = 1 # A, load transient
ΔVo_load = 0.05 # V, load transient
Vref = 0.8 # V, reference voltage
Ton_min = 90e-9 # s, Minimum duty cycle
Toff_min = 220e-9 # s, Minimum duty cycle

# Duty cycle equations

## Frequency foldback
Calculate input voltage range for the AP62301 to operate without frequency foldback.

In [37]:
Dmin_dt = Ton_min * fsw
Dmax_dt = 1 - Toff_min * fsw
VinMax_dt = Vout / Dmin_dt
VinMin_dt = Vout / Dmax_dt

In [38]:
D = (Vout + Vd)/(Vin + Vd)
print(f"VinMax_dt = {VinMax_dt:.2f} V, VinMin_dt = {VinMin_dt:.2f} V, Dmin_dt = {Dmin_dt:.3f}, Dmax_dt = {Dmax_dt:.3f}, D = {D:.3f}")

VinMax_dt = 74.07 V, VinMin_dt = 5.99 V, Dmin_dt = 0.068, Dmax_dt = 0.835, D = 0.417


# Ripple current
This is typically 30% to 60% of the maximum output current.  Ripple current versus inductor value has inverse relationship.  For 20% current ripple, a very large inductor is needed (see WE DC/DC converter handbook).  The converter might no longer regulate properly when the ripple current is too low.

For low output voltage ripple, you can always add an extra LC-filter on the output.

In [39]:
Δil_pp = r * Iout
print(f"Δil_pp = {Δil_pp:.2f} A")

Δil_pp = 0.60 A


# Inductor selection

2. Guarantee CCM operation:

During t_off, the current will drop from its peak to its minimum value.  So the difference in current over the t_off time interval is Δil_pp.  The voltage over the inductor equals the output voltage, because the bottom switch (or diode) is conducting.

$$
\begin{align*}
\left\{
    \begin {aligned}
         & V_{out} + V_{D} = L * \frac{\Delta I_{Lpp}}{t_{off}} \\
         & t_{off} = \frac{1-D}{f_{sw}}                 
    \end{aligned}
\right.
\end{align*}
\Rightarrow
L_{min} = \frac{V_{out} + V_{D}}{\Delta I_{Lpp}*f_{sw}}*(1-D)
$$
$$$$

In [40]:
L = (Vout + Vd) * (1-D)/(Δil_pp * fsw)  # see above
print(f"L = {L*1e6:.2f} µH")

L = 6.48 µH


Do this calculation for the four corner cases : Vin(min, max) and Iout(min, max).  Then take the maximum value of those four calculations.

## Rated current
Rated current is defined at DC, while in the real application, it's the maximum RMS current that will cause the heating (also taking AC-effects into account).  Check RED-expert for the real expected temperature rise.

Check the rated current for your ambient temperature situation.

## Saturation current
1. Select the maximum peak to peak current ripple in the inductor: Δil_pp.  
$$I_{peak} = I_{out} + \frac{\Delta I_{Lpp}}{2}$$
Peak current must be below saturation current at the specified temperature.  Beware that saturation current is temperature dependent.
Check RED-expert.

# Cin Input capacitor selection
## Capacitor ripple current calculation
1. Calculate RMS current (ripple current) through the input capacitor.

In [41]:
# Method 1 : WE DC/DC converter handbook, p.79
import math
I_Cin = math.sqrt( (Iout**2)/4 + ((Δil_pp**2)*D/12 ))  # from WE DC/DC converter handbook, p.79
print(f"I_Cin = {I_Cin:.2f} A")

I_Cin = 1.01 A


In [42]:
# Method 2 : Hubner
I_source = Vout * Iout / (η * Vin)
D = Vout / (Vin * η)
Iin_avg = I_source / D # 

ΔiL = (Vin - Vout) / (fsw * L) * D
Iin_pk = Iin_avg + ΔiL/2

# Input RMS current of the power stage
Iin_rms = math.sqrt(D * (Iin_pk**2 + (ΔiL**2)/3 - Iin_pk * ΔiL))

# Input RMS current of input capacitors
Icin_rms = math.sqrt(Iin_rms**2 - I_source**2)
print(f"Iin_avg = {Iin_avg:.2f} A, Iin_pk = {Iin_pk:.2f} A, Iin_rms = {Iin_rms:.2f} A, Icin_rms = {Icin_rms:.2f} A")

Iin_avg = 2.00 A, Iin_pk = 2.33 A, Iin_rms = 1.37 A, Icin_rms = 1.01 A


In [43]:
# Method 3 : Analog Devices
Icin_rms = Iout * math.sqrt(Vout * (Vin - Vout)) / Vin
print(f"Icin_rms = {Icin_rms:.2f} A")

Icin_rms = 0.99 A


## Input capacitor value calculation
In practical designs, input ripple is often limited to less than 1 % or 2 % of the input voltage to prevent controller instability.

In [44]:
ΔVin_pp = Vin * 0.01 # input voltage ripple, 1% of Vin
Cin = D * (1 - D) * Iout / (fsw * ΔVin_pp)
print(f"Cin = {Cin*1e6:.2f} µF")

Cin = 5.53 µF


2. Select a capacitor Cin:
  * with the lowest impedance at the switching frequency 
  * that can handle the ripple current at maximum ambient temperature
  * that can withstand the input voltage

Add a 100nF to 1µF ceramic capacitor in parallel to the input capacitor and place it as close as possible to the input connector.  Typically on the wire-to-board connector.  This filters out high frequency PARD spikes.  It can make a 20dB difference in measured PARD.

Group your input capacitors per type : MLCC, electrolytic, etc.
* MLCC ≤ 1µF can be skipped

For powerful regulators where multiple capacitors are needed, calculate the RMS current for each group:

In [45]:
# # Group 1
# Xcin1 = 1/ (8*fsw * Cin1)
# Zcin1 = math.sqrt(ESRmin**2 + Xcin1**2)

# # Group 2
# Xcin2 = 1/ (8*fsw * Cin2)
# Zcin2 = math.sqrt(ESRmin**2 + Xcin2**2)

# Ic1_rms = Icin_rms * Zcin2 / (Zcin1 + Zcin2)
# Ic2_rms = Icin_rms * Zcin1 / (Zcin1 + Zcin2)

# Input filter

If Zs is not much smaller than Zin, you'll have to add another capacitor (Cin2) with a high ESR.  This can be an electrolytic capacitor or an MLCC with a discrete series resistor.

See [low_pass_filter](./low_pass_filter.ipynb) for more details.

The addition of the input filter increases the output impedance of the power supply that feeds the buck regulator.  Be sure to check that the output impedance of the filter is at least 10 to 20 dB below the buck regulator's input impedance.  

For the input filter's inductive element, a ferrite bead might be more beneficial than a power inductor because of the inherent low Q of the ferrite bead.

In [46]:
# Cin1 = Cin_min
# Cin2 = 4* Cin1
# ESRmin = 0.5 * math.sqrt(Lin/Cin1)

# Zs = math.sqrt(Lin/(Cin1+Cin2))

# Damping factor, should be greater than 1

In [47]:
# import math
# # Filter impedance formed by input inductor and input capacitor

# Zs = math.sqrt(Lin/Cin_min)

# # Input impedance of the converter
# Zin = -Vin_min**2 / (Vo * Io_max)

# if(abs(Zs) > abs(Zin)):
#     print("The input filter impedance is too high")

# Output capacitor selection

The output voltage ripple is essentially composed of two parts. One part is caused by the inductor ripple current flowing through the Equivalent Series Resistance (ESR) of the output capacitors.  The other part is caused by the inductor current ripple charging and discharging the output capacitors.  The two components of the voltage ripple are not in-phase, therefore, the actual peak-to-peak ripple is less than the sum of the two peaks.

In [48]:
ΔVo_max_pp = 0.01 * Vo # V, up to 5% of Vo

# Equation set 1 with fudge factors for steady state
Co_min1 = Δil_pp / (4 * fsw * ΔVo_max_pp)
ESR_max1 = ΔVo_max_pp / (2 * Δil_pp)

# Equation set 2 with fudge factors for load transients
Co_min2 = 2 * ΔIo / (fsw * ΔVo_load)
ESR_max2 = ΔVo_load / (2 * ΔIo)

Co = max(Co_min1, Co_min2)
ESR_Co = min(ESR_max1, ESR_max2)

NameError: name 'Vo' is not defined

Group your output capacitors per type : MLCC, electrolytic, etc.

In [ ]:
# Group 1
ESR_bulk = 11e-3 # Ohm
C_bulk = 100e-6 # F
Xbulk_ESR = 1/ (8*fsw * C_bulk)
Z_bulk = math.sqrt(ESR_bulk**2 + Xbulk_ESR**2)

# Group 2
Co_MLCC = 30e-6 # F (3x10µF)
ESR_Co_MLCC = 1e-3 # Ohm (3mOhm/3)
X_Co_MLCC = 1/ (8*fsw * Co_MLCC)
Z_Co_MLCC = math.sqrt(ESR_Co_MLCC**2 + X_Co_MLCC**2)

Calculate the peak-to-peak output voltage ripple ΔVo_pp from the load current ripple : 

In [ ]:
ΔVo_pp = Δil_pp / ((1/Z_bulk) + (1/Z_Co_MLCC))

Calculate RMS output current:

In [ ]:
Ico_rms = Δil_pp / math.sqrt(12)

Add a 100nF to 1µF ceramic capacitor in parallel to the output capacitor and place it as close as possible to the load.  Typically on the wire-to-board connector.  This filters out high frequency PARD spikes.

# Feedback network design
After the resistor ratio is determined, the resistor values can be found [online](http://www.stratoloft.com/teledesign/ResistorRatio.html).

In [ ]:
# Feedback resistor ratio : Rfbt/Rfbb
rr = (Vo-Vref)/Vref
Rfbb = 3.3e3 # Ohm
Rfbt = (Vo-Vref)/Vref * Rfbb # ohm, should be in the range of 10k to 100k

# Output voltage accuracy

In [ ]:
Rfbb = 3.3e3
Rfbt = 47e3
R_tol = 1e-2 # 1% tolerance on resistors, temperature drift not considered
Rfbb_min = Rfbb * (1 - R_tol)
Rfbb_max = Rfbb * (1 + R_tol)
Rfbt_min = Rfbt * (1 - R_tol)
Rfbt_max = Rfbt * (1 + R_tol)
Vfb_min = 0.788
Vfb_nom = 0.8
Vfb_max = 0.812

Vout_min = Vfb_min * (Rfbt_min / Rfbb_max + 1)
Vout_nom = Vfb_nom * (Rfbt / Rfbb + 1)
Vout_max = Vfb_max * (Rfbt_max / Rfbb_min + 1)

print("Output voltage range: {number:.2f}V to {number2:.2f}V".format(number=Vout_min, number2=Vout_max))
print("Output voltage nominal: {number:.2f}V".format(number=Vout_nom))

Output voltage range: 11.79V to 12.61V
Output voltage nominal: 12.19V


# Under voltage lockout (UVLO) design

In [ ]:
# LMR51610
Venh = 1.227 # V
Venl = 1.00 # V
Renb = 10e3 # Ohm
Vin_rising = 40 # V

resistor_ratio = (Vin_rising-Venh) / Venh

Rent = 2*150e3 # Ohm
Vin_rising = Venh * (Rent + Renb) / Renb # V
Vin_falling = Venl * (Rent + Renb) / Renb # V

In [ ]:
# LMR16030 
## UVLO calculation
# Rent = EN pull-up resistor
# Renb = EN pull-down resistor

Vstart = 40 # [V] UVLO rising threshold
Vstop = 35 # [V] UVLO  falling threshold 
Ihys = 3.6e-6 # [A] UVLO hysteresis curr$ent
Ven = 1.2 # [V] UVLO enable threshold
Ien = 1e-6 # [A] UVLO enable current
Rent = (Vstart - Vstop) / Ihys
Renb = Ven / ((Vstart-Ven)/Rent + Ien)

# Now fill in the real resistor values and calculate the real thresholds
Rent_real = 1.3e6 # [Ohm] EN pull-up resistor
Renb_real = 39e3 # [Ohm] EN pull-down resistor
Vstart_calc = (Ven / Renb_real - Ien) * Rent_real + Ven
Vstop_calc = Vstart_calc - Rent_real * Ihys

# References
* [Tutorial by Jurgen Hubner][def]
* [Buck Power Stage Design Equations](https://www.analog.com/en/resources/app-notes/buck-power-stage-design-equations.html)

[def]: https://www.powerelectronicsnews.com/the-buck-regulator-power-supply-design-tutorial-part-2-1/
